# HR-VITON Fine-Tuning on Google Colab
**Dollaby Graduation Project**

Fine-tunes the **Try-On Condition Generator (TOCG)** of HR-VITON (ECCV 2022) on VITON-HD dataset.
- Dataset: auto-downloaded from Kaggle (`marquis03/high-resolution-viton-zalando-dataset`)
- Runtime: ~3–4 hours on Colab free T4 GPU
- Output: `tocg_finetuned.pth` saved to Google Drive
- After training: run `Colab_HR_VITON_Server.ipynb` to serve via ngrok

> **Enable GPU first:** Runtime → Change runtime type → T4 GPU

## Step 1 — Verify GPU

In [30]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✓ GPU : {name}")
print(f"✓ VRAM: {vram:.1f} GB")

if vram < 14:
    print("⚠️  Less than 14 GB VRAM — reduce BATCH_SIZE to 1 in the training cell")

## Step 2 — Mount Google Drive

In [31]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/dollaby_hrviton'
os.makedirs(f'{DRIVE}/checkpoints', exist_ok=True)
print(f"✓ Saving all outputs to: {DRIVE}")

## Step 3 — Install Dependencies & Clone HR-VITON

In [32]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'gdown', 'tqdm', 'kagglehub', 'tensorboardX'], check=True)

# Clone HR-VITON
if not os.path.exists('/content/HR-VITON'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/sangyun884/HR-VITON.git', '/content/HR-VITON'], check=True)
    print('✓ HR-VITON cloned')
else:
    print('✓ HR-VITON already cloned')

req = '/content/HR-VITON/requirements.txt'
if os.path.exists(req):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', req])
    print('✓ HR-VITON requirements installed')
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torchvision', 'scipy', 'einops', 'clean-fid', 'tqdm', 'Pillow'])
    print('✓ Core dependencies installed')

## Step 4 — Download Pretrained Checkpoints

In [33]:
import gdown, shutil

LOCAL_CKPT = '/content/HR-VITON/checkpoints'
os.makedirs(LOCAL_CKPT, exist_ok=True)

def get_checkpoint(drive_path, gdrive_id, local_name, desc):
    local = f'{LOCAL_CKPT}/{local_name}'
    if os.path.exists(drive_path):
        shutil.copy(drive_path, local)
        size = os.path.getsize(local) / 1e6
        print(f'✓ {desc} loaded from Drive cache ({size:.0f} MB)')
    else:
        print(f'Downloading {desc}...')
        gdown.download(id=gdrive_id, output=drive_path, quiet=False)
        shutil.copy(drive_path, local)
        print(f'✓ {desc} downloaded and cached to Drive')
    return local

tocg_local = get_checkpoint(
    f'{DRIVE}/checkpoints/tocg_checkpoint.pth',
    '1XJTCdRBOPVgVTmqzhVGFAgMm2NLkw5uQ',
    'tocg_checkpoint.pth',
    'TOCG (Condition Generator)'
)
gen_local = get_checkpoint(
    f'{DRIVE}/checkpoints/gen_checkpoint.pth',
    '1T5_YDUhYSSKPC_nZMk2NeC-XXUFoYeNy',
    'gen_checkpoint.pth',
    'Image Generator'
)

print(f'\n✓ Both checkpoints ready in {LOCAL_CKPT}/')

## Step 5 — Download VITON-HD Dataset from Kaggle

Uses **kagglehub** with your `KAGGLE_API_TOKEN`.

### Get your token
1. Go to **https://www.kaggle.com/settings** → API → **Create New API Token**
2. You get a token starting with `KGAT_...`
3. Paste it in the cell below when prompted — input is hidden via `getpass`

> Dataset (~13 GB) is cached in `/root/.cache/kagglehub/` — re-running is instant after first download.

In [34]:
import os
from getpass import getpass

# Input is hidden — token is never printed or stored in notebook output
if 'KAGGLE_API_TOKEN' not in os.environ or not os.environ['KAGGLE_API_TOKEN']:
    token = getpass('Paste your KAGGLE_API_TOKEN (starts with KGAT_...) — input hidden: ')
    os.environ['KAGGLE_API_TOKEN'] = token.strip()
    print('✓ KAGGLE_API_TOKEN set for this session')
else:
    print('✓ KAGGLE_API_TOKEN already set')

In [35]:
import kagglehub

print('Downloading VITON-HD Zalando dataset from Kaggle...')
print('(First run: ~13 GB download — takes ~5-10 min on Colab)')
print('(Subsequent runs: instant — served from cache)')
print()

DATASET = kagglehub.dataset_download('marquis03/high-resolution-viton-zalando-dataset')
print(f'\n✓ Dataset path: {DATASET}')

# Show top-level structure
print('\nTop-level contents:')
for item in sorted(os.listdir(DATASET)):
    full = os.path.join(DATASET, item)
    if os.path.isdir(full):
        n = len(os.listdir(full))
        print(f'  {item}/  ({n} items)')
    else:
        size = os.path.getsize(full) / 1e6
        print(f'  {item}  ({size:.1f} MB)')

In [36]:
# ── Locate the actual dataset root (handles nested zip-extracted folders) ──────
def find_dataset_root(base):
    """Walk down until we find train_pairs.txt or a 'train' directory."""
    for root, dirs, files_found in os.walk(base):
        if 'train_pairs.txt' in files_found or 'train' in dirs:
            return root
    return base

DATASET = find_dataset_root(DATASET)
print(f'✓ Dataset root: {DATASET}')

# ── Verify required folders ────────────────────────────────────────────────────
required = [
    'train/image', 'train/cloth', 'train/cloth-mask',
    'train/image-parse-v3', 'train/image-parse-agnostic-v3.2',
    'train/openpose_img', 'train/openpose_json',
    'train/image-densepose',
]

missing = [d for d in required if not os.path.exists(f'{DATASET}/{d}')]
if missing:
    print('\n⚠️  Some folders not found:')
    for m in missing:
        print(f'   {DATASET}/{m}')
    print('\nShowing actual train/ contents:')
    train_dir = f'{DATASET}/train'
    if os.path.exists(train_dir):
        for d in sorted(os.listdir(train_dir)):
            print(f'   {d}/')
    raise SystemExit('Dataset structure mismatch — check folder names above and update `required` list if needed')

n_pairs  = len(open(f'{DATASET}/train_pairs.txt').readlines())
n_images = len(os.listdir(f'{DATASET}/train/image'))
print(f'✓ Training pairs : {n_pairs:,}')
print(f'✓ Person images  : {n_images:,}')

# ── Symlink into HR-VITON project ─────────────────────────────────────────────
data_link = '/content/HR-VITON/data'
if os.path.islink(data_link) or os.path.exists(data_link):
    os.remove(data_link)
os.symlink(DATASET, data_link)
print(f'✓ Linked: {data_link} → {DATASET}')

## Step 6 — Patch train_condition.py for Fine-Tuning

In [43]:
import re, glob

SCRIPT = '/content/HR-VITON/train_condition.py'
src = open(SCRIPT).read()

# ── 1. Add --finetune_checkpoint argument ─────────────────────────────────────
if 'finetune_checkpoint' not in src:
    src = src.replace(
        'opt = parser.parse_args()',
        "parser.add_argument('--finetune_checkpoint', type=str, default='',\n"
        "                        help='Pretrained TOCG .pth to start fine-tuning from')\n"
        "    opt = parser.parse_args()"
    )
    load_block = (
        '\n    # Fine-tuning: load pretrained TOCG weights\n'
        '    if opt.finetune_checkpoint:\n'
        '        import torch as _torch\n'
        '        _state = _torch.load(opt.finetune_checkpoint, map_location=device)\n'
        '        model.load_state_dict(_state, strict=False)\n'
        '        print(f\'✓ Fine-tuning from checkpoint: {opt.finetune_checkpoint}\')\n'
    )
    src = re.sub(r'(model\.to\(device\)[^\n]*\n)', r'\1' + load_block, src, count=1)
    print('✓ Patched train_condition.py — finetune_checkpoint added')
else:
    print('✓ train_condition.py already patched (finetune_checkpoint)')

# ── 2. Fix ConditionGenerator forward call: tocg(x,y) → tocg(opt, x, y) ──────
# The model's forward(self, opt, input1, input2) requires opt as first arg
before = src.count('tocg(opt,')
src = re.sub(r'\btocg\((?!opt\b)', 'tocg(opt, ', src)
after = src.count('tocg(opt,')
if after > before:
    print(f'✓ Patched {after - before} tocg() call(s) — added opt as first argument')
else:
    print('✓ tocg() calls already patched')

open(SCRIPT, 'w').write(src)
assert 'finetune_checkpoint' in open(SCRIPT).read(), 'finetune_checkpoint patch failed'

# ── 3. Fix deprecated numpy aliases across ALL HR-VITON source files ──────────
fixed = []
for fpath in glob.glob('/content/HR-VITON/**/*.py', recursive=True):
    try:
        s = open(fpath).read()
        n = re.sub(r'\bnp\.float\b',   'float',   s)
        n = re.sub(r'\bnp\.int\b',     'int',     n)
        n = re.sub(r'\bnp\.bool\b',    'bool',    n)
        n = re.sub(r'\bnp\.complex\b', 'complex', n)
        n = re.sub(r'\bnp\.object\b',  'object',  n)
        if n != s:
            open(fpath, 'w').write(n)
            fixed.append(fpath.replace('/content/HR-VITON/', ''))
    except: pass

if fixed:
    print(f'✓ Fixed numpy aliases in {len(fixed)} file(s): {fixed}')
else:
    print('✓ No deprecated numpy aliases found')

print('\n✓ All patches applied')

In [45]:
import re
src = open('/content/HR-VITON/train_condition.py').read()
src = re.sub(r'\bmkgrid\((?!opt\b)', 'mkgrid(opt, ', src)
open('/content/HR-VITON/train_condition.py', 'w').write(src)
print('✓ Fixed mkgrid() -', src.count('mkgrid(opt,'), 'calls patched')


## Step 7 — Run Fine-Tuning

| Parameter | Fine-tuning | Original training |
|-----------|------------|-------------------|
| Batch size | **2** (T4 constraint) | 8 |
| Learning rate | **2e-5** | 2e-4 |
| Steps | **15,000** | 300,000 |
| Mixed precision | **fp32** (fp16 removed — HR-VITON lacks GradScaler) | varies |

Estimated time: **~3–4 hours** on Colab free T4

In [ ]:
import subprocess, sys, re, glob

SCRIPT = '/content/HR-VITON/train_condition.py'

src = open(SCRIPT).read()

# 1. tocg(x,y) → tocg(opt, x, y)
before = src.count('tocg(opt,')
src = re.sub(r'\btocg\((?!opt\b)', 'tocg(opt, ', src)
n_tocg = src.count('tocg(opt,') - before

# 2. mkgrid: undo wrong patch (opt first), apply correct (opt last)
src = re.sub(r'\bmkgrid\(opt,\s*(\w+),\s*(\w+),\s*(\w+)\)', r'mkgrid(\1, \2, \3, opt)', src)
before = src.count(', opt)')
src = re.sub(r'\bmkgrid\((\w+),\s*(\w+),\s*(\w+)\)', r'mkgrid(\1, \2, \3, opt)', src)
n_mkgrid = src.count(', opt)') - before

open(SCRIPT, 'w').write(src)
calls = [l.strip() for l in open(SCRIPT) if 'mkgrid(' in l]
print(f'✓ tocg: +{n_tocg}, mkgrid: +{n_mkgrid}  |  {calls}')

fixed = []
for fpath in glob.glob('/content/HR-VITON/**/*.py', recursive=True):
    try:
        s = open(fpath).read()
        n = re.sub(r'\bnp\.float\b','float',s); n = re.sub(r'\bnp\.int\b','int',n)
        n = re.sub(r'\bnp\.bool\b','bool',n);   n = re.sub(r'\bnp\.complex\b','complex',n)
        if n != s: open(fpath,'w').write(n); fixed.append(fpath.split('HR-VITON/')[-1])
    except: pass
print(f'✓ numpy: {"fixed "+str(fixed) if fixed else "clean"}')

BATCH_SIZE=2; KEEP_STEPS=15000; SAVE_FREQ=3000; VAL_FREQ=1000; LR=2e-5; WORKERS=2

cmd = ['python3', SCRIPT,'--cuda','True','--gpu_ids','0','-b',str(BATCH_SIZE),
       '-j',str(WORKERS),'--dataroot','/content/HR-VITON/data','--G_lr',str(LR),
       '--D_lr',str(LR),'--keep_step',str(KEEP_STEPS),'--save_count',str(SAVE_FREQ),
       '--val_count',str(VAL_FREQ),'--num_test_visualize','1',
       '--Ddownx2','--Ddropout','--lasttvonly','--interflowloss','--occlusion',
       '--finetune_checkpoint',f'{LOCAL_CKPT}/tocg_checkpoint.pth']

print(f'\nStarting training (batch={BATCH_SIZE}, steps={KEEP_STEPS:,}, fp32)\n')
LOG='/content/train_log.txt'
with open(LOG,'w') as log:
    proc=subprocess.Popen(cmd,cwd='/content/HR-VITON',stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in proc.stdout: sys.stdout.write(line); log.write(line)
    proc.wait()
print(f'\nTraining finished — exit code: {proc.returncode}')
if proc.returncode!=0:
    print('\nLast 30 lines:'); print(''.join(open(LOG).readlines()[-30:]))


## Step 8 — Save Fine-Tuned Checkpoint to Drive

In [41]:
import glob, shutil

all_ckpts = sorted(
    glob.glob(f'{LOCAL_CKPT}/step_*.pth') +
    glob.glob(f'{LOCAL_CKPT}/TOCG*.pth') +
    glob.glob(f'{LOCAL_CKPT}/condition*.pth'),
    key=os.path.getmtime
)
all_ckpts = [c for c in all_ckpts if 'tocg_checkpoint' not in c]

dest = f'{DRIVE}/checkpoints/tocg_finetuned.pth'

if all_ckpts:
    latest = all_ckpts[-1]
    shutil.copy(latest, dest)
    size = os.path.getsize(dest) / 1e6
    print(f'✓ Fine-tuned checkpoint saved: {dest}  ({size:.0f} MB)')
    from google.colab import files
    files.download(dest)
else:
    shutil.copy(tocg_local, dest)
    size = os.path.getsize(dest) / 1e6
    print(f'⚠️  Training did not produce a checkpoint (check Step 7 for the error).')
    print(f'   Saving pretrained TOCG to Drive as fallback: {dest}  ({size:.0f} MB)')
    print()
    print('Common causes of Step 7 failure:')
    print('  • OOM  → set BATCH_SIZE = 1 in Step 7 and re-run')
    print('  • Missing folder  → dataset path mismatch')
    print('  • Import error  → restart runtime and re-run from Step 1')

print()
print('Next steps:')
print('  → Open Colab_HR_VITON_Server.ipynb')
print('  → It will load tocg_finetuned.pth from Drive automatically')

## Step 9 — Quick Inference Test

Verifies the fine-tuned TOCG by running it on one training pair from the dataset.

In [42]:
import sys, torch, argparse, numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms

sys.path.insert(0, '/content/HR-VITON')
from networks import ConditionGenerator, load_checkpoint

device   = 'cuda'
IMG_H, IMG_W = 256, 192

opt = argparse.Namespace(
    fine_width=IMG_W, fine_height=IMG_H,
    semantic_nc=13, output_nc=13, ngf=96,
    Ddownx2=True, Ddropout=True, spectral=True,
    occlusion=True, warp_feature='T1',
    out_layer='relu',
    gen_semantic_nc=7,
    num_D=2, gpu_ids=[0],
    ndf=64, norm_D='spectralinstance', n_layers_D=3,
    norm_G='spectralaliasinstance', num_upsampling_layers='most',
    no_ganFeat_loss=False, no_vgg_loss=False,
    use_vae=False, contain_dontcare_label=False,
    crop_size=IMG_H,
    cuda=True,
)

# Load fine-tuned (or pretrained fallback)
finetuned_path = f'{DRIVE}/checkpoints/tocg_finetuned.pth'
ckpt_path      = finetuned_path if os.path.exists(finetuned_path) else tocg_local

tocg = ConditionGenerator(opt, input1_nc=4, input2_nc=16, output_nc=13, ngf=opt.ngf)
load_checkpoint(tocg, ckpt_path, opt)
tocg.to(device).eval()
print(f'✓ Loaded: {ckpt_path}')

# Read first pair
person_name, cloth_name = open(f'{DATASET}/train_pairs.txt').readline().strip().split()

tf   = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,)*3, (0.5,)*3)])
tf_l = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

person_img = Image.open(f'{DATASET}/train/image/{person_name}').convert('RGB').resize((IMG_W, IMG_H))
cloth_img  = Image.open(f'{DATASET}/train/cloth/{cloth_name}').convert('RGB').resize((IMG_W, IMG_H))
mask_img   = Image.open(f'{DATASET}/train/cloth-mask/{cloth_name}').convert('L').resize((IMG_W, IMG_H))
dense_img  = Image.open(f'{DATASET}/train/image-densepose/{person_name}').convert('RGB').resize((IMG_W, IMG_H))

parse_name = person_name.replace('.jpg', '.png')
parse_path = f'{DATASET}/train/image-parse-agnostic-v3.2/{parse_name}'
if not os.path.exists(parse_path):
    parse_path = f'{DATASET}/train/image-parse-agnostic-v3.2/{person_name}'
parse_img = Image.open(parse_path).resize((IMG_W, IMG_H), Image.NEAREST)

cloth_t = tf(cloth_img).unsqueeze(0).to(device)
mask_t  = tf_l(mask_img).unsqueeze(0).to(device)
dense_t = tf(dense_img).unsqueeze(0).to(device)

parse_np     = np.array(parse_img)
parse_onehot = torch.zeros(1, 13, IMG_H, IMG_W, device=device)
for c in range(13):
    parse_onehot[0, c] = torch.from_numpy((parse_np == c).astype(np.float32)).to(device)

input1 = torch.cat([cloth_t, mask_t], dim=1)       # [1, 4, H, W]
input2 = torch.cat([parse_onehot, dense_t], dim=1) # [1, 16, H, W]

with torch.no_grad():
    # forward(self, opt, input1, input2) → (flow_list, segmap, warped_cloth, warped_mask)
    flow_list, fake_segmap, warped_cloth, warped_mask = tocg(opt, input1, input2)

warped = ((warped_cloth.squeeze(0).cpu() + 1) / 2).clamp(0, 1)
warped_np = (warped.permute(1, 2, 0).numpy() * 255).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(12, 5))
for ax, img, title in zip(axes,
                           [person_img, cloth_img, warped_np],
                           ['Person', 'Garment', 'Warped Garment (TOCG output)']):
    ax.imshow(img); ax.set_title(title); ax.axis('off')
plt.suptitle('HR-VITON TOCG — Inference Test', fontsize=13)
plt.tight_layout()
plt.show()
print('✓ Inference test complete')